# Step 6: Model building and selection

| Step | What | How |
|---|---|---|
| 0 | Load the data, build the predictors, split into train / validation / test | |
| 1 | Check the regression assumptions | Residual plot, normal QQ plot |
| 2 | Remove predictors that repeat each other (collinearity) | F-tests (no VIF) |
| 3 | Test which predictors are significant (all are kept) | F-tests |
| 4 | Fit the full model | Linear regression |
| 5 | Create new models | Stepwise selection |
| 6 | Compare the models | Adjusted R², AIC, BIC, RMSE, MAE |
| 7 | Check the chosen model on unseen weeks | RMSE and MAE vs a simple baseline |

**Before running:** build `data/processed/model_table.parquet` with the pipeline (`scripts/01_` to `04_`, see
`CLAUDE.md`). A full run takes about 3 minutes. The full explanation of every step is in `docs/model_selection.md`.

**Main choices:**
- **Zones:** the 50 busiest zones.
- **Target:** `log(pickups + 1)` (Step 1 shows why).
- **Split:** in every month, the last 7 days are **test**, the 7 days before are **validation**, the rest is
  **train**.
- **Standard errors clustered by day:** weather is the same for every zone in a given hour, so rows from the same
  day aren't independent. All tests account for this.

## Step 0: Data and predictors

In [ ]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pandas.tseries.holiday import USFederalHolidayCalendar
from statsmodels.tools.eval_measures import meanabs, rmse

MODEL_TABLE = Path("../data/processed/model_table.parquet")
TOP_ZONES = 50        # busiest zones by training pickups
ALPHA = 0.05          # significance level
AUX_R2_FLAG = 0.8     # a predictor more than 80% explained by the others is flagged as collinear
PAIR_CORR_FLAG = 0.7  # pairs with a correlation above this are tested together
SEED = 860

INK, INK_2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "axes.edgecolor": GRID, "axes.labelcolor": INK_2,
    "text.color": INK, "xtick.color": INK_2, "ytick.color": INK_2, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left", "figure.dpi": 110,
})
pd.set_option("display.width", 160, "display.max_columns", 30, "display.float_format", "{:,.4f}".format)

**Load.** Pick the 50 busiest zones (by training pickups), add calendar columns, and add two **lags**: the pickups
in the same zone and clock hour **yesterday** (`lag24`) and **last week** (`lag168`). Both are known a day ahead, so
they can be used to plan the next day's dispatch.

Rows with a missing value are dropped: the first week of January (no "last week" yet), a few hours next to the
daylight-saving gap, and the last hour of the year (no weather).

In [ ]:
con = duckdb.connect()
df = con.execute(f"""
    WITH m AS (SELECT * FROM '{MODEL_TABLE.as_posix()}'),
    s AS (
        SELECT *, CASE WHEN day(hour) > day(last_day(hour)) - 7  THEN 'test'
                       WHEN day(hour) > day(last_day(hour)) - 14 THEN 'val'
                       ELSE 'train' END AS split
        FROM m
    ),
    top AS (
        SELECT zone FROM s WHERE split = 'train'
        GROUP BY zone ORDER BY sum(pickups) DESC LIMIT {TOP_ZONES}
    )
    SELECT s.zone, s.hour, s.split, s.pickups,
           hour(s.hour) AS hour_of_day, isodow(s.hour) AS dow, month(s.hour) AS month,
           s.hour::DATE AS date, dayofyear(s.hour) AS day,
           s.temperature_2m, s.precipitation, s.snowfall, s.snow_depth, s.cloud_cover, s.shortwave_radiation,
           s.is_day, l24.pickups AS lag24, l168.pickups AS lag168
    FROM s
    JOIN top USING (zone)
    LEFT JOIN m l24  ON l24.zone  = s.zone AND l24.hour  = s.hour - INTERVAL 24 HOUR
    LEFT JOIN m l168 ON l168.zone = s.zone AND l168.hour = s.hour - INTERVAL 168 HOUR
    ORDER BY s.zone, s.hour
""").df()

n_raw = len(df)
df = df.dropna().reset_index(drop=True)
df = df.astype({c: "float64" for c in df.columns if str(df[c].dtype) in ("Float64", "Int64")})
df["date"] = pd.to_datetime(df["date"])
print(f"{n_raw:,} zone-hours; dropped {n_raw - len(df):,} with a missing value")
print(df["split"].value_counts().reindex(["train", "val", "test"]).to_string())

**Predictors.** Each predictor is a *term*. A category such as `hour` is a group of 0/1 columns (dummies), which is
added, tested and removed as a whole.

| Term | What it is |
|---|---|
| `hour`, `dow`, `month`, `zone` | hour of day, day of week, month, pickup zone (dummies) |
| `holiday` | 1 on a US federal holiday |
| `hour_x_weekend` | a separate daily pattern for weekends |
| `zone_x_weekend` | a separate weekend level for each zone |
| `temperature_2m`, `temp_sq` | temperature, and its square (does demand rise in both cold *and* heat?) |
| `precipitation`, `snowfall`, `snow_depth`, `cloud_cover`, `shortwave_radiation`, `is_day` | weather |
| `log_lag24`, `log_lag168` | log of yesterday's and last week's pickups at the same zone and hour |

Rule: a combined term (`hour_x_weekend`, `zone_x_weekend`, `temp_sq`) is only used together with the terms it is
built from.

In [ ]:
holidays = USFederalHolidayCalendar().holidays("2025-01-01", "2025-12-31")
REF_ZONE = int(df[df["split"] == "train"].groupby("zone")["pickups"].sum().idxmax())
weekend = df["dow"] >= 6
df["holiday"] = df["date"].isin(holidays).astype(float)
df["hour_wknd"] = np.where(weekend, df["hour_of_day"], 0)                     # weekend hour, 0 on weekdays
df["zone_wknd"] = np.where(weekend & (df["zone"] != REF_ZONE), df["zone"], 0)  # weekend zone, 0 on weekdays
df["log_lag24"] = np.log1p(df["lag24"])
df["log_lag168"] = np.log1p(df["lag168"])
df["temp_sq"] = (df["temperature_2m"] - df.loc[df["split"] == "train", "temperature_2m"].mean()) ** 2

train, val, test = (df[df["split"] == s].reset_index(drop=True) for s in ["train", "val", "test"])

LOG_TARGET = "np.log1p(pickups)"
CALENDAR = ["hour", "dow", "month", "holiday"]
CONTINUOUS = ["temperature_2m", "temp_sq", "precipitation", "snowfall", "snow_depth", "cloud_cover",
              "shortwave_radiation", "is_day", "log_lag24", "log_lag168"]
INTERACTIONS = ["hour_x_weekend", "zone_x_weekend"]
ALL_TERMS = CALENDAR + ["zone"] + CONTINUOUS + INTERACTIONS
TERMS = {"hour": "C(hour_of_day)", "dow": "C(dow)", "month": "C(month)",
         "zone": f"C(zone, Treatment({REF_ZONE}))", "hour_x_weekend": "C(hour_wknd)", "zone_x_weekend": "C(zone_wknd)",
         **{t: t for t in ["holiday"] + CONTINUOUS}}
TERM_NAME = {f: t for t, f in TERMS.items()}
PARENTS = {"hour_x_weekend": ["hour", "dow"], "zone_x_weekend": ["zone", "dow"], "temp_sq": ["temperature_2m"]}

def formula(terms, target=LOG_TARGET):
    return f"{target} ~ " + (" + ".join(TERMS[t] for t in terms) if terms else "1")

def ols(terms, data=train, target=LOG_TARGET, clustered=False):
    """Linear regression. clustered=True: standard errors clustered by day (used for all tests)."""
    model = smf.ols(formula(terms, target), data)
    if clustered:
        return model.fit(cov_type="cluster", cov_kwds={"groups": data["day"]}, use_t=True)
    return model.fit()

def term_tests(res):
    """F-test per term: are all of the term's coefficients zero?"""
    t = res.wald_test_terms(skip_single=False, scalar=True).summary_frame()
    t = t.drop(index="Intercept").rename(index=TERM_NAME)
    return t.rename(columns={"P>F": "p", "df constraint": "coefficients tested"})[["coefficients tested", "F", "p"]]

def with_hierarchy(terms):
    """Remove a combined term if a term it is built from is missing."""
    terms = list(terms)
    while any(t in PARENTS and not set(PARENTS[t]) <= set(terms) for t in terms):
        terms = [t for t in terms if not (t in PARENTS and not set(PARENTS[t]) <= set(terms))]
    return terms

print(f"train {len(train):,}, validation {len(val):,}, test {len(test):,} rows")

## Step 1: Check the assumptions with a residual plot and a normal QQ plot

The tests in a regression assume that the errors (actual − predicted) have **the same spread** at every demand
level, and are roughly **normally distributed**. The model with every predictor is fitted twice, on raw `pickups`
and on `log(pickups + 1)`, and two plots are checked:

- **Residuals vs fitted.** An even band around 0 is good. A funnel shape means the errors grow (or shrink) with
  demand.
- **Normal QQ plot.** Points on the dashed line mean normally distributed errors. Points bending away at the ends
  mean more extreme errors than normal. Both axes are in standard deviations, so negative values are expected.

In [ ]:
def diagnostics(axes, fitted, resid, label):
    fitted, resid = pd.Series(np.asarray(fitted)), pd.Series(np.asarray(resid))
    s = fitted.sample(20_000, random_state=SEED).index
    ax = axes[0]
    ax.scatter(fitted[s], resid[s], s=4, alpha=0.25, color=BLUE, linewidths=0)
    ax.axhline(0, color=INK_2, lw=1)
    ax.set(title=f"{label}: residuals vs fitted", xlabel="fitted", ylabel="residual")

    ax = axes[1]
    sm.qqplot(resid.sample(5_000, random_state=SEED), fit=True, line="45", ax=ax,
              marker=".", markersize=3, alpha=0.5, markerfacecolor=BLUE, markeredgecolor=BLUE)
    ax.get_lines()[1].set(color=INK_2, linewidth=1.2, linestyle="--")
    ax.set(title=f"{label}: normal QQ plot", xlabel="theoretical quantile", ylabel="standardised residual")

fig, axes = plt.subplots(2, 2, figsize=(12, 8.5))
rows = []
for row, (target, label) in enumerate([("pickups", "Raw pickups"), (LOG_TARGET, "log(pickups + 1)")]):
    res = ols(ALL_TERMS, target=target)
    diagnostics(axes[row], res.fittedvalues, res.resid, label)
    rows.append({"target": label, "R2": res.rsquared, "share of negative predictions": (res.fittedvalues < 0).mean()})
    del res
fig.tight_layout()
plt.show()
pd.DataFrame(rows).set_index("target")

**Results.**
- **Raw pickups fail both checks.** The residual plot is a funnel (the busier the hour, the bigger the errors), the QQ
  plot bends far away on the right, and 13% of predictions are **negative pickups**.
- **`log(pickups + 1)` is much better.** The band is far more even, almost no predictions are negative (0.02%), and
  the QQ plot is closer to the line. It isn't perfect: quiet hours have somewhat larger errors, and a few hours have
  far less demand than predicted (e.g. holidays).

**Decision:** use `log(pickups + 1)`. The uneven spread that is left is handled by the day-clustered standard errors
used in every test below.

## Step 2: Remove predictors that repeat each other (collinearity, using F-tests)

**Collinearity** means two or more predictors carry the same information. The model then can't separate their effects,
so their coefficients become unreliable. Checks:

1. **Correlation** between the numeric predictors.
2. **How much of each predictor the others explain** (regress it on all other predictors and take the R²). Above
   0.8 = **flagged** as collinear.
3. **F-tests:**
   - For strongly correlated pairs (correlation above 0.7): if the pair is significant *together* but not *one at a
     time*, they share one piece of information, and the weaker one is dropped.
   - For each flagged predictor: if its own F-test is not significant, the information is already carried by the
     other predictors, so it is dropped.

In [ ]:
corr = train[CONTINUOUS].corr()
corr.round(2)

In [ ]:
rows = []
for v in CONTINUOUS:
    others = [c for c in CONTINUOUS if c != v]
    rows.append({"predictor": v,
                 "R2 explained by calendar + zone": ols(CALENDAR + ["zone"], target=v).rsquared,
                 "R2 explained by all other predictors": ols(CALENDAR + ["zone"] + others, target=v).rsquared})
aux = pd.DataFrame(rows).set_index("predictor")
aux["flagged"] = aux["R2 explained by all other predictors"] > AUX_R2_FLAG
aux

In [ ]:
res_all = ols(ALL_TERMS, clustered=True)
single = term_tests(res_all)

drop, decisions = set(), []
pairs = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()
for (a, b), r in pairs[pairs.abs() > PAIR_CORR_FLAG].items():
    joint = float(res_all.f_test(f"{a} = 0, {b} = 0").pvalue)
    pa, pb = single.loc[a, "p"], single.loc[b, "p"]
    collinear = joint < ALPHA and max(pa, pb) >= ALPHA
    weaker = a if pa >= pb else b
    if collinear:
        drop.add(weaker)
    decisions.append({"check": f"pair {a} / {b} (correlation {r:.2f})", "p together": f"{joint:.4f}",
                      "p one at a time": f"{pa:.4f} / {pb:.4f}",
                      "decision": f"drop {weaker}" if collinear else "keep both (each significant on its own)"})
for v in aux.index[aux["flagged"]]:
    p = single.loc[v, "p"]
    if p >= ALPHA:
        drop.add(v)
    decisions.append({"check": f"flagged {v} (R² {aux.loc[v, 'R2 explained by all other predictors']:.2f})",
                      "p one at a time": f"{p:.4f}",
                      "decision": f"drop {v}" if p >= ALPHA else "keep (adds information of its own)"})
del res_all

STEP2_TERMS = with_hierarchy([t for t in ALL_TERMS if t not in drop])
print("dropped for collinearity:", sorted(set(ALL_TERMS) - set(STEP2_TERMS)) or "none")
pd.DataFrame(decisions).set_index("check").fillna("")

**Results.**
- **The two lags** are strongly correlated (0.88), but each is significant on its own, so **both are kept**.
- **Five predictors are flagged** (more than 80% explained by the others): temperature, sunlight, `is_day` and both
  lags.
- **`temperature_2m` and `is_day` are dropped.** About 89% of temperature and 82% of `is_day` is explained by the
  calendar alone: season and time of day. Neither adds significant information beyond that (p = 0.07 and 0.15).
  `temp_sq` goes with temperature.
- **Sunlight and both lags are kept,** because each adds information of its own.

Note: dropping temperature only means it doesn't help *predict* demand once the calendar is known. Temperature's
effect on demand is studied separately in the FWL section of `05_statistical_tests.ipynb`.

## Step 3: Test which predictors are significant (all are kept)

Each term is tested with an F-test: "are all of this term's coefficients zero?" A group of dummies, such as the 49
zone dummies, is tested as a whole.

**Non-significant terms are kept in the model.** "Not significant" means the data can't measure the effect precisely
enough, not that the effect is zero. Removing such terms would also distort the estimates of the terms that stay,
and make their p-values look better than they are. So every predictor stays in, and its significance is reported.

In [ ]:
FULL_TERMS = list(STEP2_TERMS)
res_full = ols(FULL_TERMS, clustered=True)
tests = term_tests(res_full)
tests["significant at 5%"] = np.where(tests["p"] < ALPHA, "yes", "no (kept)")
print("not significant, kept in the model:", ", ".join(tests.index[tests["p"] >= ALPHA]) or "none")
tests.sort_values("p")

**Results.**
- **Significant:** 9 of the 14 terms: both lags, all the calendar and zone terms, and `precipitation` (p = 0.003).
- **Not significant, kept:** `holiday` (p = 0.07), `snow_depth` (0.10), `snowfall` (0.11), `shortwave_radiation`
  (0.12) and `cloud_cover` (0.28).

Why:
- **Snow:** 2025 had few snowy hours in the training days, so its effect can't be measured precisely.
- **Holidays:** only 5 holidays fall in the training days.
- **Cloud cover and sunlight:** have no detectable effect once rain, the calendar and last week's demand are known.

## Step 4: The full model

The full model contains every predictor left after Step 2, significant or not. Because the target is in logs, each
coefficient converts to a **% change in pickups** per unit of the predictor. For the lags, read it as: a 10% rise in
last week's pickups goes with roughly a (10 × coef)% rise now.

In [ ]:
print(f"full model: {len(FULL_TERMS)} terms, {len(res_full.params)} parameters, "
      f"R² = {res_full.rsquared:.4f}, adjusted R² = {res_full.rsquared_adj:.4f}, "
      f"AIC = {res_full.aic:,.1f}, BIC = {res_full.bic:,.1f}")

single_cols = [t for t in FULL_TERMS if TERMS[t] == t]
ci = res_full.conf_int().loc[single_cols]
coef = pd.DataFrame({
    "coef": res_full.params[single_cols],
    "% change per unit": 100 * np.expm1(res_full.params[single_cols]),
    "95% CI low (%)": 100 * np.expm1(ci[0]), "95% CI high (%)": 100 * np.expm1(ci[1]),
    "p": res_full.pvalues[single_cols],
    "significant at 5%": np.where(res_full.pvalues[single_cols] < ALPHA, "yes", "no (kept)")})
coef.loc[coef.index.str.startswith("log_lag"), ["% change per unit", "95% CI low (%)", "95% CI high (%)"]] = np.nan
coef

**Results.** The full model has 14 terms and **170 parameters**: R² = 0.920, adjusted R² = 0.920, AIC = 194,424.2,
BIC = 196,181.4.

| Predictor | Effect on pickups | Significant? |
|---|---|---|
| Rain | **+2.3%** per mm of rain in the hour | yes (p = 0.003) |
| Same hour last week | 10% more pickups last week → about 5.6% more now | yes |
| Same hour yesterday | 10% more pickups yesterday → about 2.4% more now | yes |
| Holiday | −13.6% (range −26% to +1%) | no (p = 0.07), kept |
| New snow | +4.5% per cm (range −1% to +10%) | no (p = 0.11), kept |
| Snow on the ground | −5% per 10 cm (range −11% to +1%) | no (p = 0.10), kept |
| Cloud cover | about −1.5% from clear to fully overcast | no (p = 0.28), kept |
| Sunlight | about −0.6% per 100 W/m² | no (p = 0.12), kept |

For the non-significant predictors, the range (95% confidence interval) includes zero: the direction makes sense,
but the data can't rule out "no effect".

## Step 5: Stepwise selection creates new models

Stepwise selection builds a series of models by adding one term at a time: at each step, the term that improves
**BIC** the most. BIC rewards a better fit but penalises every extra parameter. It stops when no term improves BIC.
Terms can also be removed if that improves BIC.

- **Forward:** starts from an empty model; every step is saved as a new model (S1, S2, ...).
- **Backward:** starts from the full model and removes terms while BIC improves.

In [ ]:
bic_cache = {}
def bic(terms):
    key = frozenset(terms)
    if key not in bic_cache:
        bic_cache[key] = ols(terms).bic
    return bic_cache[key]

def stepwise(pool, start):
    current, path = list(start), []
    best = bic(current)
    while True:
        moves = [current + [t] for t in pool if t not in current] + [[c for c in current if c != t] for t in current]
        moves = [m for m in moves if with_hierarchy(m) == m]
        if not moves:
            break
        nxt = min(moves, key=bic)
        if bic(nxt) >= best:
            break
        added, removed = set(nxt) - set(current), set(current) - set(nxt)
        change = f"+ {added.pop()}" if added else f"- {removed.pop()}"
        current, best = nxt, bic(nxt)
        path.append({"change": change, "terms": list(current), "BIC": best})
    return path

forward = stepwise(FULL_TERMS, [])
backward = stepwise(FULL_TERMS, FULL_TERMS)
for name, path in [("forward", forward), ("backward", backward)]:
    print(f"{name}: {len(path)} steps")
    for i, p in enumerate(path, 1):
        print(f"  {i:>2}. {p['change']:<22} BIC = {p['BIC']:,.0f}")

**Results.**
- **Forward:** adds the terms in this order: last week's demand, yesterday's demand, month, hour, zone, day of week,
  the weekend pattern, zone × weekend, holiday, rain, snow on the ground, sunlight, cloud cover, new snow.
- **Backward:** removes nothing.
- **New models:** this gives 14 models, **S1 to S14**. S14 is the full model.

BIC includes even the five non-significant terms. BIC treats all 228k rows as independent, so tiny improvements
count. The F-tests in Step 3 correctly treat hours on the same day as related, so they are the better guide to
significance.

## Step 6: Compare the models

| Measure | Data | Meaning | Better |
|---|---|---|---|
| Adjusted R² | train | share of demand explained, with a penalty for extra parameters | higher |
| AIC, BIC | train | fit with a penalty for extra parameters (BIC's penalty is larger) | lower |
| RMSE | validation weeks | typical prediction error in pickups per zone-hour; big misses count more | lower |
| MAE | validation weeks | average prediction error in pickups per zone-hour | lower |

The first three measure the fit on the data the model was trained on. RMSE and MAE measure predictions on weeks
the model hasn't seen, which is what matters for forecasting, so **validation RMSE picks the model**.

**Baseline:** predict each zone-hour with the pickups from the same hour last week. A useful model must beat it.

In [ ]:
models = {f"S{i}: {p['change']}": p["terms"] for i, p in enumerate(forward, 1)}
models["Full model"] = FULL_TERMS
if backward:
    models["Backward stepwise"] = backward[-1]["terms"]

names = {}  # one row per distinct model; duplicate names are joined with " = "
for name, terms in models.items():
    names.setdefault(frozenset(terms), []).append(name)

rows = []
for key, labels in names.items():
    res = ols(models[labels[0]])
    pred = np.expm1(res.predict(val)).clip(lower=0)   # back from log(pickups + 1) to pickups
    rows.append({"model": " = ".join(labels), "parameters": len(res.params), "R2_adj": res.rsquared_adj,
                 "AIC": res.aic, "BIC": res.bic,
                 "RMSE_val": rmse(pred, val["pickups"]), "MAE_val": meanabs(pred, val["pickups"])})
    del res
rows.append({"model": "Baseline: same hour last week", "RMSE_val": rmse(val["lag168"], val["pickups"]),
             "MAE_val": meanabs(val["lag168"], val["pickups"])})
table = pd.DataFrame(rows).set_index("model")

BEST = table["RMSE_val"].drop("Baseline: same hour last week").idxmin()
print(f"lowest validation RMSE: {BEST}")
table

In [ ]:
steps = table.loc[table.index.str.match(r"S\d")]
labels = steps.index.str.split(": ").str[1]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2), sharex=True)
for ax, col, title in [(axes[0], "R2_adj", "Adjusted R² (train)"), (axes[1], "BIC", "BIC (train, lower is better)"),
                       (axes[2], "RMSE_val", "RMSE on validation weeks (pickups)")]:
    ax.plot(range(len(steps)), steps[col], marker="o", ms=5, color=BLUE)
    ax.set_title(title)
    ax.set_xticks(range(len(steps)), labels, rotation=60, ha="right", fontsize=8)
base_rmse = table.loc["Baseline: same hour last week", "RMSE_val"]
axes[2].axhline(base_rmse, color=INK_2, lw=1.2, ls="--")
axes[2].annotate("last-week baseline", (0, base_rmse), xytext=(4, 4), textcoords="offset points", color=INK_2, fontsize=9)
fig.suptitle("Stepwise models: each point adds one term", x=0.01, ha="left", fontsize=12, fontweight="bold")
fig.tight_layout()
plt.show()

**Results.**
- **On the training data, the full model wins:** adjusted R² rises and AIC and BIC fall with every term added.
- **On the validation weeks, the simplest models win.** S2 (only yesterday's and last week's demand) has the lowest
  error: RMSE **29.9 pickups**, vs 33.0 for the full model and 34.5 for the baseline.
- **Why:** last week's demand at the same zone and hour already contains that zone's usual pattern for that weekday
  and hour. The calendar and zone terms make the training fit slightly better, but that doesn't carry over to new
  weeks. The weather terms change the error by less than 0.1 pickups.

**Decision:**
- **S2 is the forecasting model:** lowest error on unseen weeks.
- **The full model is the explanatory model:** use it to describe how each factor (rain, holidays, snow, ...)
  affects demand.

## Step 7: Check the chosen model on the test weeks

The chosen model is refitted on train + validation and scored once on the **test** weeks, which no earlier step
has used. Memorial Day, Thanksgiving and Christmas fall in test weeks.

In [ ]:
chosen = models[BEST.split(" = ")[0]]
res_final = ols(chosen, data=pd.concat([train, val], ignore_index=True))
print(formula(chosen))
test["model"] = np.expm1(res_final.predict(test)).clip(lower=0)

summary = pd.DataFrame({
    "RMSE": [rmse(test["model"], test["pickups"]), rmse(test["lag168"], test["pickups"])],
    "MAE": [meanabs(test["model"], test["pickups"]), meanabs(test["lag168"], test["pickups"])],
}, index=[f"chosen model ({BEST})", "baseline: same hour last week"])
by_zone = test.groupby("zone").apply(lambda g: pd.Series({
    "mean pickups": g["pickups"].mean(), "RMSE model": rmse(g["model"], g["pickups"]),
    "RMSE baseline": rmse(g["lag168"], g["pickups"])}), include_groups=False)
by_zone["model wins"] = by_zone["RMSE model"] < by_zone["RMSE baseline"]
print(f"model beats the baseline in {by_zone['model wins'].mean():.0%} of the {len(by_zone)} zones")
summary

In [ ]:
by_zone.sort_values("mean pickups", ascending=False).head(15)

In [ ]:
zone = by_zone["mean pickups"].idxmax()
week = test[(test["zone"] == zone) & (test["hour"].dt.month == 6)].set_index("hour")
fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(week.index, week["pickups"], color=BLUE, label="actual", lw=2)
ax.plot(week.index, week["model"], color=ORANGE, label="chosen model", lw=1.6)
ax.plot(week.index, week["lag168"], color=AQUA, label="same hour last week", lw=1.6)
ax.set(title=f"Zone {zone}, June test week: hourly pickups", ylabel="pickups per hour")
ax.legend(frameon=False, ncols=3, loc="upper left")
fig.tight_layout()
plt.show()

**Results.**
- **Test weeks:** the chosen model (S2) has RMSE **34.4** and MAE **19.3** pickups per zone-hour. The baseline has
  41.1 and 22.4, so the model's typical error is **16% smaller**.
- **By zone:** it beats the baseline in **all 50 zones**.
- **Why test errors are higher than validation errors:** the test weeks include holidays, when last week's demand
  is a poor guide.
- **Zone 161 (Midtown Center), June test week:** the model follows the daily cycle closely, but it can't anticipate
  unusual peaks.

## Notes for the report

- **Two models, two questions.** The full model explains *why* demand changes (rain +2.3% per mm, etc.). S2
  forecasts demand best. Say which one each number comes from.
- **The forecasting model assumes it is used one day ahead,** since it needs yesterday's demand.
- **Demand = completed pickups.** People who couldn't find a taxi aren't counted, so busy hours are probably
  underestimated.